# 08 - Monitoring: did the forecasts come true?

<!-- contract -->

| | |
|---|---|
| **Reads** | `flight_delay_predictions` |
| **Writes** | `prediction_monitoring` |
| **Runtime** | seconds |
| **Requires** | `07_score` run at least twice on the same flight: once before it departed, once after it arrived |

Every other notebook in this project measures the model against a held-out slice of the
same 2019-2023 extract it was trained on. This one measures it against flights that did not
exist when it was trained, scored by the artifact actually carrying `@champion`, and checked
against what the aircraft did.

## What this is not

It is **not a retraining set**, and the difference is worth stating because the obvious next
move is to treat it as one.

The live path yields a handful of flights per run against a 400-unit monthly quota. Set
beside 2,463,979 Gold rows, adding one percent to the training set would take years. Volume
is not even the real objection: these rows are whichever flights someone typed into a widget,
so they are a biased sample of one or two routes. Retraining on them would pull the model
toward those routes and call it improvement.

The same rows are valuable for the question they *can* answer. A model whose calibrator was
fitted on 2022 and whose threshold was selected on 2022 is being asked to hold up years
later, and nothing else in this repo checks whether it does.

## The three questions

1. **Is it still calibrated?** Of the flights it called 30% likely, were about 30% late? The
   isotonic stage corrected this on the 2023 test window; calibration is a property of a
   distribution, and the distribution has moved on.
2. **Which cut was right?** The F1 cut and the advisory cut disagree by design. Now there are
   outcomes, so the disagreement is scorable rather than arguable.
3. **Does it fail when the NAS is degraded?** `07_score` records FAA airspace conditions at
   prediction time and the model has never seen them. If the misses concentrate under active
   conditions, that is the evidence that would justify collecting NAS history and building
   the feature. If they do not, the caption stays a caption.

None of these is answerable from one flight. The table accumulates so that they become
answerable, and this notebook says how far off that is rather than pretending otherwise.

In [0]:
import sys
sys.path.append("..")

import matplotlib.pyplot as plt
from pyspark.sql import functions as F

from src import config, plotting as P

print(f"Reading  {config.PREDICTIONS}")
print(f"Writing  {config.MONITORING}")

Reading  workspace.flights.flight_delay_predictions
Writing  workspace.flights.prediction_monitoring


## Resolved predictions

A prediction is *resolved* once `arrival_delay` is filled in, which happens when a later run
of `06_api_ingest` re-fetches the flight after it has landed and the MERGE overwrites its row
with the revised arrival time. Until then the forecast is outstanding, not wrong, and it is
excluded rather than counted as a miss.

**One hazard, handled explicitly.** `07_score` keys its MERGE on
`(airline_code, fl_number, flight_date, scoring_date)`, so re-scoring the same flight on the
same day overwrites the row rather than adding one. If that re-score happens after pushback,
`dep_delay` now exists, the flight routes to the in-flight model, and the stored probability
is no longer the pre-departure forecast a traveller was shown. Pooling the two would credit
the weak model with the strong model's accuracy.

`recommended_model` records which variant produced the stored number, so every metric below
is reported per variant and never pooled. The pre-departure rows are the ones that matter:
they are the only ones that were a forecast rather than an observation.

In [0]:
preds = spark.table(config.PREDICTIONS)

resolved = (
    preds
    .filter(F.col("arrival_delay").isNotNull())
    # The label the model was trained on: US DOT counts an arrival as delayed at
    # 15 minutes. Same definition in Silver, same definition here, or the numbers
    # are not comparable with anything else in the project.
    .withColumn("actually_delayed", (F.col("arrival_delay") >= 15).cast("int"))
    .withColumn("p", F.col("delay_probability_pct") / 100.0)
    .withColumn("correct_f1", (F.col("will_be_delayed") == F.col("actually_delayed")).cast("int"))
    .withColumn("correct_advisory",
                (F.col("advisory_flag") == F.col("actually_delayed")).cast("int"))
    .withColumn("nas_active", F.col("nas_conditions").isNotNull())
)

total = preds.count()
n = resolved.count()
outstanding = total - n

print(f"{total:,} predictions written, {n:,} resolved, {outstanding:,} still outstanding.")
if n == 0:
    print()
    print("Nothing to measure yet. A prediction resolves when the flight lands and")
    print("06_api_ingest re-fetches it, so the sequence is:")
    print()
    print("  1. Run 06 + 07 before departure   -> the forecast is written")
    print("  2. Wait for the flight to arrive")
    print("  3. Run 06 + 07 again, same flight -> arrival_delay fills in, and the")
    print("     row becomes one measurement in this table")
    print()
    print("Every cell below is a no-op until step 3 has happened at least once.")

18 predictions written, 18 resolved, 0 still outstanding.


## Accuracy, per variant and per cut

Four numbers, and the honest reading of them depends entirely on `n`. At single digits this
is a listing of individual flights that happens to be formatted as a table; the sample column
is there so nobody reads a rate off three rows.

In [0]:
if n:
    by_variant = (
        resolved.groupBy("recommended_model")
        .agg(
            F.count("*").alias("flights"),
            F.sum("actually_delayed").alias("actually_late"),
            F.avg("correct_f1").alias("accuracy_f1_cut"),
            F.avg("correct_advisory").alias("accuracy_advisory_cut"),
            F.avg("p").alias("mean_predicted"),
            F.avg(F.col("actually_delayed").cast("double")).alias("observed_rate"),
            # Brier on the probability actually stored, which after 05_train's
            # fix is the calibrated one the threshold was selected against.
            F.avg(F.pow(F.col("p") - F.col("actually_delayed").cast("double"), 2)).alias("brier"),
        )
        .orderBy("recommended_model")
        .toPandas()
    )
    display(by_variant)

    for _, row in by_variant.iterrows():
        variant, flights = row["recommended_model"], int(row["flights"])
        print(f"\n{variant}  ({flights} resolved flight(s))")
        if flights < config.MONITORING_MIN_SAMPLE:
            print(f"  Below {config.MONITORING_MIN_SAMPLE}. These are individual outcomes, not")
            print(f"  a rate -- one more flight moves 'accuracy' by "
                  f"{1 / max(flights, 1):.0%}. Reported so the table is not empty,")
            print(f"  not because {flights} flights measure anything.")
        else:
            print(f"  accuracy at the F1 cut       {row['accuracy_f1_cut']:.3f}")
            print(f"  accuracy at the advisory cut {row['accuracy_advisory_cut']:.3f}")
            print(f"  mean predicted {row['mean_predicted']:.3f} vs observed "
                  f"{row['observed_rate']:.3f}  (gap "
                  f"{row['mean_predicted'] - row['observed_rate']:+.3f})")
            print(f"  Brier {row['brier']:.4f}")
            print("  A positive gap is over-forecasting: the model is more pessimistic")
            print("  than the airline turned out to be.")

recommended_model,flights,actually_late,accuracy_f1_cut,accuracy_advisory_cut,mean_predicted,observed_rate,brier
in_flight,3,0,0.6666666666666666,0.3333333333333333,0.421700316805216,0.0,0.31206768203753166
pre_departure,15,0,0.2,0.2,0.21904876996061656,0.0,0.048414854626038895



in_flight  (3 resolved flight(s))
  Below 30. These are individual outcomes, not
  a rate -- one more flight moves 'accuracy' by 33%. Reported so the table is not empty,
  not because 3 flights measure anything.

pre_departure  (15 resolved flight(s))
  Below 30. These are individual outcomes, not
  a rate -- one more flight moves 'accuracy' by 7%. Reported so the table is not empty,
  not because 15 flights measure anything.


## Calibration drift

`05_train` fitted the isotonic calibrator on 2022 and confirmed it on 2023. Calibration is a
property of a distribution rather than of a model, so it survives only as long as the
distribution does. This is the check that would catch it decaying — and the one that needs
the most data before it says anything, because ten bins over a small sample is noise with a
line through it.

In [0]:
if n >= config.MONITORING_MIN_SAMPLE:
    bins = (
        resolved
        .withColumn("bin", F.least(F.floor(F.col("p") * 10), F.lit(9)))
        .groupBy("bin")
        .agg(F.avg("p").alias("mean_predicted"),
             F.avg(F.col("actually_delayed").cast("double")).alias("observed_rate"),
             F.count("*").alias("flights"))
        .orderBy("bin").toPandas()
    )

    fig, ax = P.new_axes((6.5, 6))
    ax.plot([0, 1], [0, 1], color=P.MUTED, linestyle="--", linewidth=1.4, label="perfect")
    ax.plot(bins["mean_predicted"], bins["observed_rate"], color=P.ACCENT,
            marker="o", markersize=7, linewidth=2.2, label="live flights")
    for _, r in bins.iterrows():
        ax.annotate(f"n={int(r['flights'])}", (r["mean_predicted"], r["observed_rate"]),
                    textcoords="offset points", xytext=(6, -10), fontsize=7, color=P.MUTED)
    ax.legend(frameon=False, fontsize=9)
    P.style(ax, title="Calibration on flights the model was never trained on",
            xlabel="Predicted P(delayed)", ylabel="Observed delay rate")
    plt.tight_layout(); plt.show()

    gap = float((bins["observed_rate"] - bins["mean_predicted"]).abs().max())
    print(f"Largest bin deviation: {gap:.4f}")
    print("Compare against the figure 05_train reported on the 2023 test window.")
    print("Materially worse here means the calibrator has aged and should be refitted")
    print("on recent data -- which is a far cheaper fix than retraining the model.")
elif n:
    print(f"{n} resolved flight(s); {config.MONITORING_MIN_SAMPLE} needed before a")
    print("reliability diagram is worth drawing. Ten bins over this many flights is")
    print("a handful per bin, and the curve would move on every new arrival.")
    print()
    print("Stating the threshold and refusing to plot below it is the point. A")
    print("reliability diagram drawn on eight flights looks exactly as authoritative")
    print("as one drawn on eight thousand.")

18 resolved flight(s); 30 needed before a
reliability diagram is worth drawing. Ten bins over this many flights is
a handful per bin, and the curve would move on every new arrival.

Stating the threshold and refusing to plot below it is the point. A
reliability diagram drawn on eight flights looks exactly as authoritative
as one drawn on eight thousand.


## The NAS question

The one measurement here that could change the model rather than just grade it.

`07_score` records FAA airspace conditions at prediction time, and the model has never seen
them — there is no historical NAS archive, so the column cannot be built for 2019-2023 and a
model that never saw it cannot be scored on it. That is why it is a caption.

If misses concentrate on flights predicted while a ground delay programme or ground stop was
in force, the caption is load-bearing and the fix is to start collecting NAS status daily
until there is enough history to train on. If they do not concentrate, then nothing here
justifies the cost, and saying so is the same result.

In [0]:
if n:
    nas = (
        resolved.groupBy("nas_active")
        .agg(F.count("*").alias("flights"),
             F.avg("correct_f1").alias("accuracy"),
             F.avg("p").alias("mean_predicted"),
             F.avg(F.col("actually_delayed").cast("double")).alias("observed_rate"))
        .orderBy("nas_active").toPandas()
    )
    display(nas)

    groups = {bool(r["nas_active"]): r for _, r in nas.iterrows()}
    if len(groups) == 2 and all(int(r["flights"]) >= config.MONITORING_MIN_SAMPLE
                                for r in groups.values()):
        quiet, active = groups[False], groups[True]
        # Under-forecasting is predicted < observed, so a *more negative* gap under
        # active conditions is the model missing delays the airspace explains.
        gap_quiet = float(quiet["mean_predicted"] - quiet["observed_rate"])
        gap_active = float(active["mean_predicted"] - active["observed_rate"])
        print(f"\nForecast gap, quiet airspace : {gap_quiet:+.3f}")
        print(f"Forecast gap, active conditions: {gap_active:+.3f}")
        if gap_active < gap_quiet - 0.05:
            print("\nThe model under-forecasts more when the NAS is degraded. That is the")
            print("blind spot showing up where it was predicted to, and it is the argument")
            print("for collecting NAS status daily until a feature can be built.")
        else:
            print("\nNo material difference. On this evidence the airspace feed earns its")
            print("place as context for a reader and nothing more -- which is where it is.")
    else:
        print(f"\nNot enough resolved flights in both groups "
              f"({config.MONITORING_MIN_SAMPLE} each) to compare. This is the slowest")
        print("question to answer here: it needs flights predicted while conditions were")
        print("active, and most days most airports report nothing.")

nas_active,flights,accuracy,mean_predicted,observed_rate
true,18,0.2777777777777778,0.2528240277680498,0.0



Not enough resolved flights in both groups (30 each) to compare. This is the slowest
question to answer here: it needs flights predicted while conditions were
active, and most days most airports report nothing.


## Persist

One row per resolved prediction, overwritten each run because it is derived entirely from
`flight_delay_predictions` and nothing is gained by making it incremental. The model version
travels with the row: a champion replaced by a retrain is a different model, and pooling its
outcomes with its predecessor's would hide exactly the regression this table exists to catch.

In [0]:
if n:
    monitoring = resolved.select(
        "scoring_date", "prediction_timestamp", "ingest_run_id",
        "flight", "route", "flight_date", "origin_airport_code", "destination_airport_code",
        "recommended_model", "basis",
        F.col("p").alias("predicted_probability"),
        "applied_threshold", "advisory_threshold",
        "will_be_delayed", "advisory_flag",
        "dep_delay", "arrival_delay", "actually_delayed",
        "correct_f1", "correct_advisory",
        "nas_active", "nas_conditions",
        # Which artifact made the call. Without this, a retrain silently pools two
        # different models' outcomes into one accuracy number.
        "model_pre", "model_pre_version", "model_in", "model_in_version",
    )

    (monitoring.write.format("delta").mode("overwrite")
     .option("overwriteSchema", "true").saveAsTable(config.MONITORING))
    print(f"{config.MONITORING}: {monitoring.count():,} resolved prediction(s)")
    display(monitoring.orderBy(F.desc("prediction_timestamp")).limit(20))
else:
    print(f"Nothing resolved, so {config.MONITORING} was not written.")
    print("An empty monitoring table would imply the question had been asked and")
    print("answered. It has not been asked yet.")

workspace.flights.prediction_monitoring: 18 resolved prediction(s)


scoring_date,prediction_timestamp,ingest_run_id,flight,route,flight_date,origin_airport_code,destination_airport_code,recommended_model,basis,predicted_probability,applied_threshold,advisory_threshold,will_be_delayed,advisory_flag,dep_delay,arrival_delay,actually_delayed,correct_f1,correct_advisory,nas_active,nas_conditions,model_pre,model_pre_version,model_in,model_in_version
2026-09-16,2026-09-16T20:05:20.140Z,20260916T163927Z,UA3987.0,ATL -> IAH,2026-09-16,ATL,IAH,pre_departure,pre-departure (schedule only),0.2432260121134842,0.19,0.19,1,1,0.0,0.0,0,0,0,true,UNKNOWN,workspace.flights.rf_pre_departure,9,workspace.flights.rf_in_flight,7
2026-09-16,2026-09-16T20:05:20.140Z,20260916T164956Z,UA1211.0,IAH -> ATL,2026-09-16,IAH,ATL,pre_departure,pre-departure (schedule only),0.22486036996771705,0.19,0.19,1,1,0.0,0.0,0,0,0,true,UNKNOWN,workspace.flights.rf_pre_departure,9,workspace.flights.rf_in_flight,7
2026-09-16,2026-09-16T20:05:20.140Z,20260916T163927Z,UA2249.0,ATL -> IAH,2026-09-16,ATL,IAH,in_flight,in-flight (departure delay known),0.05263157894736842,0.41,0.07,0,0,6.0,0.0,0,1,1,true,UNKNOWN,workspace.flights.rf_pre_departure,9,workspace.flights.rf_in_flight,7
2026-09-16,2026-09-16T20:05:20.140Z,20260916T163927Z,DL1223.0,ATL -> IAH,2026-09-16,ATL,IAH,pre_departure,pre-departure (schedule only),0.22486036996771705,0.19,0.19,1,1,0.0,0.0,0,0,0,true,UNKNOWN,workspace.flights.rf_pre_departure,9,workspace.flights.rf_in_flight,7
2026-09-16,2026-09-16T20:05:20.140Z,20260916T164956Z,DL1682.0,IAH -> ATL,2026-09-16,IAH,ATL,pre_departure,pre-departure (schedule only),0.1784872236028803,0.19,0.19,0,0,0.0,0.0,0,1,1,true,UNKNOWN,workspace.flights.rf_pre_departure,9,workspace.flights.rf_in_flight,7
2026-09-16,2026-09-16T20:05:20.140Z,20260916T193404Z,F90.0,ATL -> LAX,2026-09-16,ATL,LAX,pre_departure,pre-departure (schedule only),0.18100567342814952,0.19,0.19,0,0,67.0,0.0,0,1,1,true,LAX: closure (closed May 27 at 18:26 UTC. reopens May 28 at 16:00 UTC. — !LAX 05/277 LAX AD AP CLSD TO NON SKED TRANSIENT GA ACFT EXC 24HR PPR CTC ATLANTIC AVIATION 310-258-9884 OR SIGNATURE AVIATION 310-410-9605 2605271826-2705281600),workspace.flights.rf_pre_departure,9,workspace.flights.rf_in_flight,7
2026-09-16,2026-09-16T20:05:20.140Z,20260916T163927Z,DL1572.0,ATL -> IAH,2026-09-16,ATL,IAH,pre_departure,pre-departure (schedule only),0.18800107613666936,0.19,0.19,0,0,0.0,0.0,0,1,1,true,UNKNOWN,workspace.flights.rf_pre_departure,9,workspace.flights.rf_in_flight,7
2026-09-16,2026-09-16T20:05:20.140Z,20260916T164956Z,UA1228.0,IAH -> ATL,2026-09-16,IAH,ATL,pre_departure,pre-departure (schedule only),0.23264984227129337,0.19,0.19,1,1,0.0,0.0,0,0,0,true,UNKNOWN,workspace.flights.rf_pre_departure,9,workspace.flights.rf_in_flight,7
2026-09-16,2026-09-16T20:05:20.140Z,20260916T193404Z,DL301.0,ATL -> LAX,2026-09-16,ATL,LAX,pre_departure,pre-departure (schedule only),0.22486036996771705,0.19,0.19,1,1,0.0,0.0,0,0,0,true,LAX: closure (closed May 27 at 18:26 UTC. reopens May 28 at 16:00 UTC. — !LAX 05/277 LAX AD AP CLSD TO NON SKED TRANSIENT GA ACFT EXC 24HR PPR CTC ATLANTIC AVIATION 310-258-9884 OR SIGNATURE AVIATION 310-410-9605 2605271826-2705281600),workspace.flights.rf_pre_departure,9,workspace.flights.rf_in_flight,7
2026-09-16,2026-09-16T20:05:20.140Z,20260916T163927Z,DL1131.0,ATL -> IAH,2026-09-16,ATL,IAH,pre_departure,pre-departure (schedule only),0.2432260121134842,0.19,0.19,1,1,0.0,0.0,0,0,0,true,UNKNOWN,workspace.flights.rf_pre_departure,9,workspace.flights.rf_in_flight,7


## What this notebook decided

| Decision | Method | Defence |
|---|---|---|
| Live rows are monitoring, not training data | Documented in `config.MONITORING`, never joined back into Gold | A few flights a month, self-selected by whoever typed the widget, against 2.4M training rows: retraining on them biases the model toward one route and calls it improvement |
| Outcome definition | `arrival_delay >= 15`, the US DOT rule | The same definition Silver labels with; any other makes these numbers incomparable with every other figure in the project |
| Variants never pooled | Grouped by `recommended_model` | Re-scoring after pushback replaces the pre-departure forecast with an in-flight one; pooling credits the 0.63 model with the 0.93 model's accuracy |
| Small samples refuse to plot | `MONITORING_MIN_SAMPLE`, stated and enforced | A reliability diagram drawn on eight flights looks as authoritative as one drawn on eight thousand |
| Model version on every row | `model_*_version` carried through | A retrained champion is a different model; pooling its outcomes with its predecessor's hides the regression this table exists to find |
| NAS checked, not assumed | Accuracy split by whether conditions were active at prediction time | The project rejected NAS as a feature for a stated reason; this is the measurement that would overturn that, and it is allowed to come back negative |